<a href="https://colab.research.google.com/github/srishtisrii/PatentTrendAI-Automated-Patent-Trend-Detection-and-Forecasting/blob/main/Copy_of_3rd_colab_file_made_with_mock_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[CELL-1] Install all required libraries
Wait for this to fully finish before running anything else.
 You will see a lot of output scrolling — that is normal.

In [ ]:
!pip install vaderSentiment prophet plotly sentence-transformers scikit-learn nltk --quiet

[CELL-2] All imports

In [ ]:
import warnings
import logging
import re
import random
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from prophet import Prophet
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)

print("All imports successful.")


[CELL-3] Generate mock patent data (Step 1 + 2)

In [ ]:
random.seed(42)
np.random.seed(42)

TITLE_TEMPLATES = [
    "System and method for {task} using {tech}",
    "Neural network architecture for {task} in {domain}",
    "Apparatus and method for {task} based on {tech}",
    "{tech}-based approach for {task} optimization",
    "Deep learning framework for {task} in {domain}",
    "Machine learning pipeline for automated {task}",
    "Transformer model for {task} across {domain} applications",
    "Federated learning system for {task} with privacy constraints",
    "Attention mechanism for {task} using {tech}",
    "Reinforcement learning approach for {domain} {task}",
]

TASKS = [
    "image classification", "object detection", "semantic segmentation",
    "natural language understanding", "text generation", "speech recognition",
    "anomaly detection", "predictive maintenance", "recommendation systems",
    "knowledge graph construction", "question answering",
    "named entity recognition", "sentiment analysis",
    "document summarization", "code generation",
    "drug discovery", "protein structure prediction",
    "fraud detection", "autonomous navigation", "time-series forecasting",
]

TECHS = [
    "convolutional neural networks", "transformer architectures",
    "recurrent neural networks", "graph neural networks",
    "generative adversarial networks", "variational autoencoders",
    "attention mechanisms", "self-supervised learning",
    "federated learning", "reinforcement learning",
    "knowledge distillation", "neural architecture search",
]

DOMAINS = [
    "medical imaging", "autonomous vehicles",
    "natural language processing", "computer vision",
    "robotics", "financial services", "cybersecurity",
    "edge computing", "healthcare", "manufacturing",
]

ABSTRACT_SENTENCES = [
    "The present invention relates to a system and method for {task} utilizing {tech}.",
    "A novel {tech} architecture is disclosed for achieving improved performance in {domain} applications.",
    "The disclosed system processes input data through multiple layers of {tech} to generate accurate outputs.",
    "Training is performed on large-scale datasets using gradient-based optimization techniques.",
    "The invention achieves state-of-the-art results on standard {domain} benchmarks.",
    "An encoder-decoder framework based on {tech} is employed for feature extraction and prediction.",
    "The system incorporates attention mechanisms to selectively weight relevant input features.",
    "Hardware accelerators including GPUs and specialized processors are leveraged for efficient inference.",
    "The method significantly reduces computational overhead while maintaining prediction accuracy.",
    "Applications include real-time processing in {domain} environments with limited computational resources.",
    "Experimental evaluations demonstrate significant improvements over prior art methods.",
    "The system is deployable on both cloud infrastructure and on-device edge platforms.",
    "A multi-layer architecture enables hierarchical feature extraction from raw input signals.",
    "The invention further discloses a training procedure that mitigates overfitting on limited datasets.",
    "Outputs are post-processed using calibration techniques to produce reliable probability estimates.",
]

ASSIGNEES = [
    "Google LLC", "Microsoft Technology Licensing LLC",
    "International Business Machines Corporation", "Amazon Technologies Inc",
    "Apple Inc", "Meta Platforms Inc", "Samsung Electronics Co Ltd",
    "Qualcomm Incorporated", "Intel Corporation", "NVIDIA Corporation",
    "Baidu Inc", "Huawei Technologies Co Ltd", "Sony Corporation",
    "Siemens AG", "Robert Bosch GmbH", "General Electric Company",
    "Ford Global Technologies LLC", "Toyota Motor Corporation",
    "University of California", "Massachusetts Institute of Technology",
]

start_date = datetime(2018, 1, 1)
end_date   = datetime(2024, 6, 30)
date_span  = (end_date - start_date).days
records    = []

for i in range(500):
    task   = random.choice(TASKS)
    tech   = random.choice(TECHS)
    domain = random.choice(DOMAINS)

    title    = random.choice(TITLE_TEMPLATES).format(task=task, tech=tech, domain=domain).title()
    abstract = " ".join(
        s.format(task=task, tech=tech, domain=domain)
        for s in random.sample(ABSTRACT_SENTENCES, random.randint(6, 10))
    )

    records.append({
        "patent_id": f"MOCK{10000000 + i}",
        "title":     title,
        "abstract":  abstract,
        "date":      pd.Timestamp(start_date + timedelta(days=random.randint(0, date_span))),
        "assignee":  random.choice(ASSIGNEES) if random.random() > 0.10 else "",
    })

df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Mock data generated: {len(df)} records")
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Avg abstract length: {int(df['abstract'].str.len().mean())} characters")


[CELL-4] Preprocessing (Step 3)

In [ ]:
BASE_SW    = set(stopwords.words("english"))
PATENT_SW  = {
    "invention", "present", "disclose", "disclosed", "discloses",
    "method", "system", "apparatus", "device", "arrangement",
    "claim", "claims", "embodiment", "embodiments",
    "example", "examples", "use", "using", "used", "uses",
    "provide", "provided", "provides", "include", "includes", "including",
    "relate", "relates", "related", "describe", "described", "describes",
    "according", "based", "one", "two", "three", "four", "five",
    "plurality", "least", "first", "second", "third",
    "thereby", "wherein", "whereby", "herein", "thereof",
    "thereto", "therefrom", "said", "hereinafter",
    "preferably", "substantially", "generally", "particularly",
    "additionally", "furthermore", "moreover", "thus", "therefore",
}
STOPWORDS = BASE_SW | PATENT_SW

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text   = text.lower()
    text   = re.sub(r"[^a-z\s]", " ", text)
    text   = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df["cleaned_abstract"] = df["abstract"].apply(clean_text)
df["cleaned_title"]    = df["title"].apply(clean_text)

empty = (df["cleaned_abstract"].str.len() == 0).sum()
if empty > 0:
    df = df[df["cleaned_abstract"].str.len() > 0].copy().reset_index(drop=True)
    print(f"Dropped {empty} rows with empty cleaned abstracts.")

print(f"Preprocessing done: {len(df)} records")
print(f"Avg tokens after cleaning: {df['cleaned_abstract'].str.split().str.len().mean():.1f}")


[CELL-5] TF-IDF Vectorization (Step 4)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2),
    min_df       = 2,
    max_df       = 0.90,
    sublinear_tf = True,
)
tfidf_matrix  = vectorizer.fit_transform(df["cleaned_abstract"])
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF matrix: {tfidf_matrix.shape[0]} patents × {tfidf_matrix.shape[1]} terms")



[CELL-6] SBERT Embeddings (Step 5)
This takes 30–90 seconds. A progress bar will appear.

In [ ]:
print("Loading SBERT model (downloads ~90MB on first run)...")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. Encoding abstracts...")

embeddings = sbert_model.encode(
    df["abstract"].tolist(),
    batch_size        = 32,
    show_progress_bar = True,
    convert_to_numpy  = True,
)

assert embeddings.shape == (len(df), 384)
assert not np.isnan(embeddings).any()

np.save("embeddings.npy", embeddings)
print(f"Embeddings shape: {embeddings.shape} — saved to embeddings.npy")


[CELL-7] K-Means Clustering (Step 6)
 Using k=5 directly — elbow analysis was already done in the previous session.
 Change OPTIMAL_K here if your earlier elbow plot suggested a different value.


In [ ]:
OPTIMAL_K = 5

print(f"Fitting KMeans with k={OPTIMAL_K}...")

km_final     = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(embeddings)

def get_top_keywords(cluster_id, top_n=8):
    mask        = (df["cluster"].values == cluster_id)
    mean_tfidf  = tfidf_matrix[mask].toarray().mean(axis=0)
    top_indices = mean_tfidf.argsort()[::-1][:top_n]
    return [feature_names[i] for i in top_indices]

cluster_label_map = {}
print("\n=== Cluster Keywords ===")
for cid in range(OPTIMAL_K):
    kws                    = get_top_keywords(cid, top_n=8)
    cluster_label_map[cid] = " | ".join(kws[:3])
    count                  = (df["cluster"] == cid).sum()
    print(f"Cluster {cid} ({count} patents): {kws}")

df["cluster_label"] = df["cluster"].map(cluster_label_map)
assert df["cluster_label"].isna().sum() == 0

print(f"\nAll {len(df)} patents assigned a cluster and label.")


[CELL-8] VADER Sentiment Analysis (Step 7)

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return round(analyzer.polarity_scores(text)["compound"], 4)

df["sentiment_score"] = df["abstract"].apply(get_sentiment)

sentiment_summary = (
    df.groupby(["cluster", "cluster_label"])["sentiment_score"]
    .agg(mean_sentiment="mean", median_sentiment="median", patent_count="count")
    .round(4)
    .reset_index()
    .sort_values("cluster")
    .reset_index(drop=True)
)

print("=== Innovation Positivity Score per Cluster ===\n")
for _, row in sentiment_summary.iterrows():
    print(f"Cluster {int(row['cluster'])} | {row['cluster_label']}")
    print(f"  Mean: {row['mean_sentiment']:.4f}  |  Patents: {int(row['patent_count'])}")


[CELL-9] Sentiment Bar Chart (Step 7 visualization)

In [ ]:
COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]

def hex_to_rgba(hex_color, alpha):
    h    = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

fig_sentiment = go.Figure()
fig_sentiment.add_trace(go.Bar(
    x            = [f"Cluster {int(r['cluster'])}" for _, r in sentiment_summary.iterrows()],
    y            = sentiment_summary["mean_sentiment"],
    text         = sentiment_summary["mean_sentiment"].round(3).astype(str),
    textposition = "outside",
    marker_color = COLORS[:OPTIMAL_K],
    customdata   = sentiment_summary[["cluster_label", "patent_count"]],
    hovertemplate= (
        "<b>%{x}</b><br>"
        "Label: %{customdata[0]}<br>"
        "Mean Sentiment: %{y:.4f}<br>"
        "Patents: %{customdata[1]}<extra></extra>"
    ),
))
fig_sentiment.update_layout(
    title         = "Innovation Positivity Score by Technology Cluster",
    xaxis_title   = "Cluster",
    yaxis_title   = "Mean VADER Compound Score",
    yaxis         = dict(range=[-0.1, 0.8]),
    plot_bgcolor  = "white",
    paper_bgcolor = "white",
    font          = dict(size=13),
    height        = 500,
)
fig_sentiment.update_xaxes(showgrid=False)
fig_sentiment.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_sentiment.write_html("sentiment_chart.html")
fig_sentiment.show()

print("Sentiment chart saved to sentiment_chart.html")


[CELL-10] Prophet Forecasting (Step 8)
 Fits one model per cluster. Takes ~30–60 seconds total.

In [ ]:
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

full_month_range = pd.date_range(
    start = df["month"].min(),
    end   = df["month"].max(),
    freq  = "MS"
)

print(f"Fitting Prophet for {OPTIMAL_K} clusters "
      f"({df['month'].min().date()} → {df['month'].max().date()})...\n")

forecasts = {}

for cid in range(OPTIMAL_K):
    cluster_df = df[df["cluster"] == cid].copy()

    monthly = (
        cluster_df.groupby("month").size()
        .reset_index(name="y")
        .rename(columns={"month": "ds"})
    )
    full_df  = pd.DataFrame({"ds": full_month_range})
    monthly  = full_df.merge(monthly, on="ds", how="left").fillna(0)
    monthly["y"] = monthly["y"].astype(float)

    model = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        interval_width          = 0.95,
        changepoint_prior_scale = 0.05,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(monthly)

    future   = model.make_future_dataframe(periods=24, freq="MS")
    forecast = model.predict(future)

    forecast = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
    forecast["yhat"]       = forecast["yhat"].clip(lower=0).round(3)
    forecast["yhat_lower"] = forecast["yhat_lower"].clip(lower=0).round(3)
    forecast["yhat_upper"] = forecast["yhat_upper"].clip(lower=0).round(3)
    forecast["cluster"]    = cid
    forecast["is_future"]  = forecast["ds"] > monthly["ds"].max()

    forecasts[cid] = {"model": model, "forecast": forecast, "history": monthly}
    print(f"  Cluster {cid} done — forecast to {forecast['ds'].max().date()}")

print("\nAll Prophet models fitted.")


[CELL-11] Forecast Chart (Step 8 visualization)

In [ ]:
fig_forecast = make_subplots(
    rows             = OPTIMAL_K,
    cols             = 1,
    shared_xaxes     = True,
    vertical_spacing = 0.06,
    subplot_titles   = [
        f"Cluster {cid}: {df[df['cluster']==cid]['cluster_label'].iloc[0]}"
        for cid in range(OPTIMAL_K)
    ],
)

for cid in range(OPTIMAL_K):
    fc      = forecasts[cid]["forecast"]
    history = forecasts[cid]["history"]
    color   = COLORS[cid]
    row     = cid + 1
    cutoff  = history["ds"].max()
    hist_fc = fc[fc["ds"] <= cutoff]
    fut_fc  = fc[fc["ds"] >  cutoff]

    fig_forecast.add_trace(go.Scatter(
        x         = pd.concat([hist_fc["ds"], hist_fc["ds"][::-1]]),
        y         = pd.concat([hist_fc["yhat_upper"], hist_fc["yhat_lower"][::-1]]),
        fill      = "toself",
        fillcolor = hex_to_rgba(color, 0.15),
        line      = dict(width=0),
        name      = "95% CI",
        showlegend= (cid == 0),
        hoverinfo = "skip",
    ), row=row, col=1)

    fig_forecast.add_trace(go.Bar(
        x            = history["ds"],
        y            = history["y"],
        name         = "Actual count",
        marker_color = hex_to_rgba(color, 0.40),
        showlegend   = (cid == 0),
    ), row=row, col=1)

    fig_forecast.add_trace(go.Scatter(
        x          = hist_fc["ds"],
        y          = hist_fc["yhat"],
        mode       = "lines",
        line       = dict(color=color, width=2),
        name       = "Fitted trend",
        showlegend = (cid == 0),
    ), row=row, col=1)

    fig_forecast.add_trace(go.Scatter(
        x          = fut_fc["ds"],
        y          = fut_fc["yhat"],
        mode       = "lines",
        line       = dict(color=color, width=2, dash="dash"),
        name       = "24-month forecast",
        showlegend = (cid == 0),
    ), row=row, col=1)

    fig_forecast.add_vline(
        x          = cutoff.timestamp() * 1000,
        line_width = 1,
        line_dash  = "dot",
        line_color = "grey",
        row=row, col=1,
    )

fig_forecast.update_layout(
    title         = "Patent Filing Trends and 24-Month Forecast by Technology Cluster",
    height        = 300 * OPTIMAL_K,
    showlegend    = True,
    plot_bgcolor  = "white",
    paper_bgcolor = "white",
    font          = dict(size=12),
)
fig_forecast.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_forecast.update_yaxes(showgrid=True, gridcolor="lightgrey", title_text="Patents / month")
fig_forecast.write_html("forecast_chart.html")
fig_forecast.show()

print("Forecast chart saved to forecast_chart.html")


[CELL-12] Save all output files

In [ ]:
df.to_csv("patents_final.csv", index=False)
sentiment_summary.to_csv("sentiment_summary.csv", index=False)

all_fc_rows = []
for cid, data in forecasts.items():
    fc    = data["forecast"].copy()
    fc["cluster_label"] = df[df["cluster"] == cid]["cluster_label"].iloc[0]
    all_fc_rows.append(fc)

forecasts_df = pd.concat(all_fc_rows, ignore_index=True)
forecasts_df.to_csv("forecasts.csv", index=False)

print("=== All files saved ===")
print(f"  patents_final.csv     — {len(df)} rows, {len(df.columns)} columns")
print(f"  sentiment_summary.csv — {len(sentiment_summary)} rows")
print(f"  forecasts.csv         — {len(forecasts_df)} rows")
print(f"  embeddings.npy        — shape {embeddings.shape}")
print(f"  sentiment_chart.html")
print(f"  forecast_chart.html")
print()
print("Full pipeline complete. Ready for Step 9 — Streamlit Dashboard.")


# [CELL DRIVE-1] Mount Google Drive
 **A popup will appear asking you to sign in and give permission. Allow it.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted at /content/drive")

[CELL DRIVE-2] Create the project folder and save all files

In [ ]:
import os
import shutil

# All files will live here in your Google Drive
DRIVE_FOLDER = "/content/drive/MyDrive/PatentTrendAI"
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f"Folder ready: {DRIVE_FOLDER}")

# List of every file the pipeline produces
FILES_TO_SAVE = [
    "patents_final.csv",
    "patents_preprocessed.csv",
    "sentiment_summary.csv",
    "forecasts.csv",
    "embeddings.npy",
    "sentiment_chart.html",
    "forecast_chart.html",
    "cluster_selection_plots.png",
]

saved = []
missing = []

for filename in FILES_TO_SAVE:
    src  = f"/content/{filename}"
    dst  = f"{DRIVE_FOLDER}/{filename}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        saved.append(filename)
        print(f"  Saved  {filename:<35} ({size_kb:.1f} KB)")
    else:
        missing.append(filename)
        print(f"  SKIP   {filename:<35} (not found in /content/)")

print()
print(f"Saved  : {len(saved)} files")
if missing:
    print(f"Missing: {missing}  (run the relevant pipeline cells first)")
else:
    print("All files saved successfully.")
print()
print("Your files are now permanently stored in Google Drive.")
print("They will survive any Colab runtime reset.")


[CELL DRIVE-3] Fast recovery loader
 ─────────────────────────────────────────────────────────────────────────────
 WHEN TO USE THIS CELL:
   Any time your Colab runtime resets and you need files back quickly.
   Run DRIVE-1 first (mount Drive), then run this cell.
   It restores all CSV files to /content/ in under 10 seconds —
   no re-running SBERT, Prophet, or any other slow step.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive", force_remount=True)

DRIVE_FOLDER = "/content/drive/MyDrive/PatentTrendAI"

FILES_TO_RESTORE = [
    "patents_final.csv",
    "patents_preprocessed.csv",
    "sentiment_summary.csv",
    "forecasts.csv",
    "embeddings.npy",
    "sentiment_chart.html",
    "forecast_chart.html",
]

print("Restoring files from Google Drive...\n")
restored = []
missing  = []

for filename in FILES_TO_RESTORE:
    src = f"{DRIVE_FOLDER}/{filename}"
    dst = f"/content/{filename}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        restored.append(filename)
        print(f"  Restored  {filename:<35} ({size_kb:.1f} KB)")
    else:
        missing.append(filename)
        print(f"  MISSING   {filename:<35} (not in Drive yet)")

print()
print(f"Restored : {len(restored)} files")
if missing:
    print(f"Missing  : {missing}")
    print("  → Run the full pipeline to regenerate missing files, then re-run DRIVE-2.")
else:
    print("All files restored. You can now run any downstream cell directly.")



 TASK 1 — Run Streamlit Dashboard from Colab
 =============================================================================
 Add these cells to the BOTTOM of your current Colab file, after CELL-12.

 WHAT THIS DOES:
   Saves dashboard.py to /content/, starts Streamlit, and gives you a
   public URL to open the dashboard in your browser — no local setup needed.

 HOW IT WORKS:
   Uses Colab's built-in port proxy — no ngrok account, no localtunnel,
   no external service required.


 [CELL DASH-1] Install Streamlit
 (Already installed if you ran CELL-1 earlier in the same session.
  If not, uncomment the line below and run it.)

In [ ]:
!pip install streamlit --quiet

[CELL DASH-2] Write dashboard.py to disk
 This cell writes the complete dashboard code as a file in /content/.
 It is self-contained — paste the entire cell as-is.





In [ ]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

st.set_page_config(
    page_title="PatentTrendAI", page_icon="🔬",
    layout="wide", initial_sidebar_state="expanded"
)

COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

@st.cache_data
def load_data():
    df = pd.read_csv("patents_final.csv", parse_dates=["date"])
    df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
    sentiment_summary = pd.read_csv("sentiment_summary.csv")
    forecasts_df = pd.read_csv("forecasts.csv", parse_dates=["ds"])
    return df, sentiment_summary, forecasts_df

try:
    df, sentiment_summary, forecasts_df = load_data()
except FileNotFoundError as e:
    st.error(f"Missing file: {e}. Make sure all CSV files are in /content/.")
    st.stop()

OPTIMAL_K = df["cluster"].nunique()
cluster_label_map = (
    df[["cluster","cluster_label"]].drop_duplicates()
    .set_index("cluster")["cluster_label"].to_dict()
)

with st.sidebar:
    st.title("🔬 PatentTrendAI")
    st.markdown("**Automated Patent Trend Detection and Forecasting**")
    st.markdown("---")
    st.markdown("**Project Info**")
    st.markdown("- **Student**: Srishti Srivastava")
    st.markdown("- **Roll No**: 221030346")
    st.markdown("- **Org**: GreyB")
    st.markdown("---")
    selected_cluster = st.selectbox(
        "Focus Cluster:",
        options=sorted(df["cluster"].unique()),
        format_func=lambda x: f"Cluster {x}: {cluster_label_map[x]}"
    )

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Overview", "🔍 Cluster Analysis", "📈 Forecast", "💡 Sentiment", "📋 Explorer"
])

with tab1:
    st.header("Dataset Overview")
    c1,c2,c3,c4,c5 = st.columns(5)
    c1.metric("Total Patents", f"{len(df):,}")
    c2.metric("Clusters", OPTIMAL_K)
    c3.metric("Date Range", f"{df[\'date\'].min().year}–{df[\'date\'].max().year}")
    c4.metric("Assignees", f"{df[df[\'assignee\']!=\'\'][\'assignee\'].nunique():,}")
    c5.metric("Avg Sentiment", f"{df[\'sentiment_score\'].mean():.3f}")
    st.markdown("---")
    cl, cr = st.columns(2)
    with cl:
        dist = df["cluster"].value_counts().sort_index()
        fig = go.Figure(go.Pie(
            labels=[f"C{i}: {cluster_label_map[i]}" for i in dist.index],
            values=dist.values, hole=0.4,
            marker_colors=COLORS[:OPTIMAL_K],
            textinfo="label+percent"
        ))
        fig.update_layout(title="Cluster Distribution", height=400, showlegend=False, plot_bgcolor="white")
        st.plotly_chart(fig, use_container_width=True)
    with cr:
        monthly_all = df.groupby(["month","cluster"]).size().reset_index(name="count")
        fig2 = go.Figure()
        for cid in range(OPTIMAL_K):
            cd = monthly_all[monthly_all["cluster"]==cid]
            fig2.add_trace(go.Scatter(x=cd["month"],y=cd["count"],mode="lines",
                name=f"C{cid}",line=dict(color=COLORS[cid],width=2)))
        fig2.update_layout(title="Monthly Filings by Cluster",height=400,
            plot_bgcolor="white",paper_bgcolor="white",
            legend=dict(orientation="h",yanchor="bottom",y=1.02))
        fig2.update_xaxes(showgrid=True,gridcolor="lightgrey")
        fig2.update_yaxes(showgrid=True,gridcolor="lightgrey")
        st.plotly_chart(fig2, use_container_width=True)
    st.subheader("Top 15 Assignees")
    top_a = df[df["assignee"]!=""]["assignee"].value_counts().head(15).reset_index()
    top_a.columns=["assignee","count"]
    fig3 = px.bar(top_a,x="count",y="assignee",orientation="h",
        color="count",color_continuous_scale="Blues")
    fig3.update_layout(height=420,plot_bgcolor="white",coloraxis_showscale=False,
        yaxis=dict(autorange="reversed"))
    st.plotly_chart(fig3, use_container_width=True)

with tab2:
    st.header(f"Cluster {selected_cluster} — Deep Dive")
    cdf = df[df["cluster"]==selected_cluster]
    m1,m2,m3 = st.columns(3)
    m1.metric("Patents in Cluster", len(cdf))
    m2.metric("Mean Sentiment", f"{cdf[\'sentiment_score\'].mean():.4f}")
    m3.metric("Unique Assignees", cdf[cdf["assignee"]!=""]["assignee"].nunique())
    ca, cb = st.columns(2)
    with ca:
        st.subheader("Top Keywords in Titles")
        tw = cdf["cleaned_title"].str.split().explode().value_counts().head(15).reset_index()
        tw.columns=["keyword","freq"]
        fig_kw = px.bar(tw,x="freq",y="keyword",orientation="h",
            color_discrete_sequence=[COLORS[selected_cluster]])
        fig_kw.update_layout(height=380,plot_bgcolor="white",
            yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig_kw, use_container_width=True)
    with cb:
        st.subheader("Top Assignees in Cluster")
        ta = cdf[cdf["assignee"]!=""]["assignee"].value_counts().head(10).reset_index()
        ta.columns=["assignee","count"]
        fig_ta = px.bar(ta,x="count",y="assignee",orientation="h",
            color_discrete_sequence=[COLORS[selected_cluster]])
        fig_ta.update_layout(height=380,plot_bgcolor="white",
            yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig_ta, use_container_width=True)
    st.subheader("Monthly Trend — This Cluster")
    cm = cdf.groupby("month").size().reset_index(name="count")
    fig_cm = go.Figure(go.Bar(x=cm["month"],y=cm["count"],
        marker_color=hex_to_rgba(COLORS[selected_cluster],0.6)))
    fig_cm.update_layout(height=280,plot_bgcolor="white",paper_bgcolor="white")
    fig_cm.update_yaxes(showgrid=True,gridcolor="lightgrey")
    st.plotly_chart(fig_cm, use_container_width=True)

with tab3:
    st.header("24-Month Patent Filing Forecast")
    fig_fc = make_subplots(rows=OPTIMAL_K,cols=1,shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[f"C{cid}: {cluster_label_map[cid]}" for cid in range(OPTIMAL_K)])
    for cid in range(OPTIMAL_K):
        fc = forecasts_df[forecasts_df["cluster"]==cid].copy()
        color = COLORS[cid]; row = cid+1
        cutoff = fc[fc["is_future"]==False]["ds"].max()
        hist_fc = fc[fc["ds"]<=cutoff]; fut_fc = fc[fc["ds"]>cutoff]
        hc = df[df["cluster"]==cid].groupby("month").size().reset_index(name="y").rename(columns={"month":"ds"})
        fig_fc.add_trace(go.Scatter(
            x=pd.concat([hist_fc["ds"],hist_fc["ds"][::-1]]),
            y=pd.concat([hist_fc["yhat_upper"],hist_fc["yhat_lower"][::-1]]),
            fill="toself",fillcolor=hex_to_rgba(color,0.12),line=dict(width=0),
            name="95% CI",showlegend=(cid==0),hoverinfo="skip"),row=row,col=1)
        fig_fc.add_trace(go.Bar(x=hc["ds"],y=hc["y"],
            marker_color=hex_to_rgba(color,0.4),name="Actual",showlegend=(cid==0)),row=row,col=1)
        fig_fc.add_trace(go.Scatter(x=hist_fc["ds"],y=hist_fc["yhat"],mode="lines",
            line=dict(color=color,width=2),name="Fitted",showlegend=(cid==0)),row=row,col=1)
        fig_fc.add_trace(go.Scatter(x=fut_fc["ds"],y=fut_fc["yhat"],mode="lines",
            line=dict(color=color,width=2,dash="dash"),name="Forecast",showlegend=(cid==0)),row=row,col=1)
        if pd.notna(cutoff):
            fig_fc.add_vline(x=cutoff.timestamp()*1000,line_width=1,
                line_dash="dot",line_color="grey",row=row,col=1)
    fig_fc.update_layout(height=280*OPTIMAL_K,showlegend=True,
        plot_bgcolor="white",paper_bgcolor="white",font=dict(size=11))
    fig_fc.update_xaxes(showgrid=True,gridcolor="lightgrey")
    fig_fc.update_yaxes(showgrid=True,gridcolor="lightgrey",title_text="Patents/month")
    st.plotly_chart(fig_fc, use_container_width=True)
    st.subheader("Forecast Summary")
    future_only = forecasts_df[forecasts_df["is_future"]==True]
    rows = []
    for cid in range(OPTIMAL_K):
        cfc = future_only[future_only["cluster"]==cid]
        rows.append({"Cluster":cid,"Label":cluster_label_map[cid],
            "Avg/month":f"{cfc[\'yhat\'].mean():.2f}",
            "Peak/month":f"{cfc[\'yhat\'].max():.2f}",
            "Until":cfc["ds"].max().strftime("%b %Y")})
    st.dataframe(pd.DataFrame(rows),use_container_width=True,hide_index=True)

with tab4:
    st.header("Innovation Positivity Score")
    cs1,cs2 = st.columns([3,2])
    with cs1:
        fig_s = go.Figure(go.Bar(
            x=[f"C{int(r[\'cluster\'])}" for _,r in sentiment_summary.iterrows()],
            y=sentiment_summary["mean_sentiment"],
            text=sentiment_summary["mean_sentiment"].round(3).astype(str),
            textposition="outside",marker_color=COLORS[:OPTIMAL_K],
            customdata=sentiment_summary[["cluster_label","patent_count"]],
            hovertemplate="<b>%{x}</b><br>%{customdata[0]}<br>Score: %{y:.4f}<br>Patents: %{customdata[1]}<extra></extra>"
        ))
        fig_s.update_layout(title="Mean VADER Score per Cluster",
            yaxis=dict(range=[-0.1,0.8]),plot_bgcolor="white",paper_bgcolor="white",height=400)
        st.plotly_chart(fig_s, use_container_width=True)
    with cs2:
        disp = sentiment_summary[["cluster","cluster_label","mean_sentiment","patent_count"]].copy()
        disp.columns=["Cluster","Label","Score","Patents"]
        disp["Score"]=disp["Score"].round(4)
        st.dataframe(disp,use_container_width=True,hide_index=True)
        fig_h = px.histogram(df,x="sentiment_score",color="cluster",nbins=40,
            barmode="overlay",opacity=0.6,color_discrete_sequence=COLORS[:OPTIMAL_K])
        fig_h.update_layout(height=280,plot_bgcolor="white",paper_bgcolor="white")
        st.plotly_chart(fig_h, use_container_width=True)

with tab5:
    st.header("Patent Explorer")
    fe1,fe2,fe3 = st.columns(3)
    with fe1:
        cf = st.multiselect("Cluster",sorted(df["cluster"].unique()),
            default=sorted(df["cluster"].unique()),format_func=lambda x:f"C{x}")
    with fe2:
        ymin=int(df["date"].dt.year.min()); ymax=int(df["date"].dt.year.max())
        yr = st.slider("Year Range",ymin,ymax,(ymin,ymax))
    with fe3:
        srch = st.text_input("Search Title","")
    filt = df[(df["cluster"].isin(cf))&(df["date"].dt.year>=yr[0])&(df["date"].dt.year<=yr[1])]
    if srch:
        filt = filt[filt["title"].str.lower().str.contains(srch.lower(),na=False)]
    st.markdown(f"**{len(filt):,} of {len(df):,} patents**")
    dcols = ["patent_id","title","date","assignee","cluster","cluster_label","sentiment_score"]
    dd = filt[dcols].copy()
    dd["date"]=dd["date"].dt.strftime("%Y-%m-%d")
    dd["sentiment_score"]=dd["sentiment_score"].round(4)
    dd.columns=["ID","Title","Date","Assignee","Cluster","Label","Sentiment"]
    st.dataframe(dd,use_container_width=True,hide_index=True,height=450)

st.markdown("---")
st.caption("PatentTrendAI | Srishti Srivastava (221030346) | GreyB | Final Year B.Tech CSE-AI")
'''

with open("/content/dashboard.py", "w") as f:
    f.write(dashboard_code)

print("dashboard.py written to /content/dashboard.py")
print("File size:", len(dashboard_code), "characters")


[CELL DASH-3] Start Streamlit and get the public URL

 Run this cell and then click the URL that appears.
 The dashboard opens in a new browser tab.

In [ ]:
import subprocess
import threading
import time

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/dashboard.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Give Streamlit a few seconds to start before showing the URL
time.sleep(5)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print()
print("=" * 60)
print("DASHBOARD IS RUNNING")
print("=" * 60)
print(f"Open this URL in your browser:")
print(f"  {url}")
print()
print("The URL changes every time you restart — always run this")
print("cell again after a reconnect to get the current URL.")
print("=" * 60)
